In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus/CLI-Demos
/home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'mp4', 'dicom', 'nifti']


In [5]:
scan_type = 'dicom'

scan_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/DICOM/UCSD/OrganizedDICOM/P07/V02/IM_0362.dcm'
scan_loader_kwargs = {
    'transpose': False,
}

In [6]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)
# bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

In [7]:
image_data.pixel_data.shape

(2047, 600, 800, 3)

In [10]:
## CEUS

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

ref_slice = image_data.pixel_data[:, :, 100, 0]
axial_slice = image_data.pixel_data[:, :, 100, 50]

# Mask out zeros before computing percentiles
non_zero = axial_slice[axial_slice != 0]
non_zero_ref = ref_slice[ref_slice != 0]

p_low = np.mean(non_zero_ref)
p_high = np.percentile(non_zero, 100)

# Clip and normalize to 0-255
clipped_slice = np.clip(axial_slice, p_low, p_high)
normalized_slice = ((clipped_slice - p_low) / (p_high - p_low) * 255).astype(np.uint8)

# Display all three side by side
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

axes[0].imshow(axial_slice.T, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
plt.colorbar(axes[0].images[0], ax=axes[0], label='Pixel Intensity')

axes[1].imshow(normalized_slice.T, cmap='gray')
axes[1].set_title(f'Clipped [{p_low:.1f}, {p_high:.1f}] → Normalized [0, 255]')
axes[1].axis('off')
plt.colorbar(axes[1].images[0], ax=axes[1], label='Pixel Intensity')

plt.tight_layout()
plt.show()

## Image Preprocessing

In [ ]:
from src.image_preprocessing.options import get_im_preproc_funcs, get_required_im_preproc_kwargs

print("Available preprocessing functions:", list(get_im_preproc_funcs().keys()))

In [ ]:
preproc_func_names = ['resample'] # in order of application
required_kwargs = get_required_im_preproc_kwargs(preproc_func_names)
print("Required kwargs for preprocessing functions:", required_kwargs)

In [ ]:
preproc_kwargs = {
    'target_vox_size': (1.0, 1.0, 1.0),
    'interp': 'linear',
}

In [ ]:
from src.entrypoints import scan_preprocessing_step

image_data = scan_preprocessing_step(preproc_func_names, image_data, **preproc_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [11]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [14]:
seg_type = 'nifti'

seg_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/UCSD/P03/UCSD-P03-V02-CE2-MC_VOI.nii.gz'
seg_loader_kwargs = {}

In [15]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## Segmentation Preprocessing

In [ ]:
from src.seg_preprocessing.options import get_seg_preproc_funcs, get_required_seg_preproc_kwargs

print("Available preprocessing functions:", list(get_seg_preproc_funcs().keys()))

In [ ]:
preproc_func_names = ['resample'] # in order of application
required_kwargs = get_required_seg_preproc_kwargs(preproc_func_names)
print("Required kwargs for preprocessing functions:", required_kwargs)

In [ ]:
preproc_kwargs = {
    'target_vox_size': (1.0, 1.0, 1.0),
    'interp': 'nearest',
}

In [ ]:
from src.entrypoints import seg_preprocessing_step

seg_data = seg_preprocessing_step(preproc_func_names, image_data, seg_data, **preproc_kwargs)

## CEUS Quantitative Temporal Curve Analysis

In [16]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))

Available analysis types: ['curves_paramap', 'curves']


In [17]:
analysis_type = 'curves'

print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis functions: ['tic', 'pyradiomics']


In [18]:
analysis_funcs = ['tic']

# Find all required kwargs for the analysis functions
analysis_funcs = analysis_funcs if len(analysis_funcs) else list(all_analysis_funcs[analysis_type].keys())
required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: []


In [20]:
analysis_kwargs = {
    'curves_output_path': '/media/ahmed-el-kaffas/20TB-HDD/Farah /3DMPUS/RESULTS/UCSD-P03/ucsd-p03-v02-ce2-curves.csv',
}

In [21]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

Computing curves: 100%|██████████| 450/450 [00:55<00:00,  8.14it/s]


## Curve Quantification

In [22]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['lognormal_fit_full', 'dte', 'wash_rates', 'first_order_full', 'auc_no_fit', 'cmus_firstorder', 'first_order_select', 'lognormal_fit_select'])


In [24]:
function_names = ['dte'] # Empty list will use all functions
output_path = '/media/ahmed-el-kaffas/20TB-HDD/Farah /3DMPUS/RESULTS/UCSD-P03/ucsd-p03-v02-ce2-quant.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['moderate-pselectin_diagnostics_Image-original_Mean'],
    'n_frames_to_analyze': 100,
    'tic_name': 'TIC'
}

In [25]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)